# lcz4r_python — function tests
Run cells top-to-bottom. Each section is independent after `map_path` is set in §1.

In [9]:
import sys, os
sys.path.insert(0, os.getcwd())

from lcz_get_map import lcz_get_map, lcz_clear_cache
from lcz_get_map_euro import lcz_get_map_euro
from lcz_get_map_usa import lcz_get_map_usa
from lcz_get_map_generator import lcz_get_map_generator
from lcz_plot_map import lcz_plot_map
from lcz_cal_area import lcz_cal_area
from lcz_get_parameters import lcz_get_parameters
from lcz_plot_parameters import lcz_plot_parameters
from lcz_get_ucp import lcz_get_ucp


## 1 — lcz_get_map (global)

In [10]:
map_path = lcz_get_map(city="Aracaju", isave_map=False, lang = "pt")
print('map_path:', map_path)

11:18:29 - _lcz_map_engine - INFO - Loading clipped map from local cache.


map_path: /Users/co2map/.lcz4r_cache/clipped_d358344d777633e4.tif


## 2 — lcz_get_map_euro

In [3]:
euro_path = lcz_get_map_euro(city="Lisbon", isave_map=False)
print('euro_path:', euro_path)

euro_path: /Users/co2map/.lcz4r_cache/clipped_11dd96e2c42452bb.tif


## 3 — lcz_get_map_usa

In [4]:
usa_path = lcz_get_map_usa(city="Miami", isave_map=False)
print('usa_path:', usa_path)

usa_path: /Users/co2map/.lcz4r_cache/clipped_350c95c0f75a957c.tif


## 4 — lcz_get_map_generator

In [5]:
gen_path = lcz_get_map_generator(id = "3110e623fbe4e73b1cde55f0e9832c4f5640ac21")
print('generator path:', gen_path)

generator path: /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmp4nxoa5k9.tif


## 5 — lcz_plot_map
Uses `map_path` from §1.

In [6]:
# Plotly renderer (default) — inline in notebook
result_ml = lcz_plot_map(map_path, renderer="maplibre", opacity=0.8, isave=True, lang = "zh")
result_ml.show()

# MapLibre renderer — OSM basemap + LCZ overlay + opacity slider
# result_ml = lcz_plot_map(map_path, renderer="maplibre", opacity=0.8, isave=True)
# result_ml.show()

## 6 — lcz_cal_area

In [4]:
# Bar chart (default)
area = lcz_cal_area(map_path, plot_type="sunburst", isave=True, lang = "zh")
print(area.df)
area.fig.show()

shape: (15, 7)
┌─────┬───────┬────────────┬───────────┬────────────┬─────────┬────────────────┐
│ lcz ┆ count ┆ area_km2   ┆ area_perc ┆ lcz_name   ┆ lcz_col ┆ lcz_colorblind │
│ --- ┆ ---   ┆ ---        ┆ ---       ┆ ---        ┆ ---     ┆ ---            │
│ i16 ┆ u64   ┆ f32        ┆ f32       ┆ str        ┆ str     ┆ str            │
╞═════╪═══════╪════════════╪═══════════╪════════════╪═════════╪════════════════╡
│ 1   ┆ 710   ┆ 5.3        ┆ 0.87      ┆ 紧凑高层   ┆ #910613 ┆ #E16A86        │
│ 2   ┆ 1669  ┆ 12.46      ┆ 2.05      ┆ 紧凑中层   ┆ #D9081C ┆ #D8755E        │
│ 3   ┆ 9242  ┆ 68.989998  ┆ 11.33     ┆ 紧凑低层   ┆ #FF0A22 ┆ #C98027        │
│ 4   ┆ 132   ┆ 0.99       ┆ 0.16      ┆ 开放高层   ┆ #C54F1E ┆ #B48C00        │
│ 5   ┆ 17801 ┆ 133.020004 ┆ 21.84     ┆ 开放中层   ┆ #FF6628 ┆ #989600        │
│ …   ┆ …     ┆ …          ┆ …         ┆ …          ┆ …       ┆ …              │
│ 12  ┆ 2274  ┆ 16.99      ┆ 2.79      ┆ 疏林       ┆ #00A926 ┆ #009EDA        │
│ 14  ┆ 3476  ┆ 25.98      ┆ 4.27  

In [8]:
# Sunburst (hierarchical Urban / Natural / Other)
lcz_cal_area(map_path, plot_type="treemap").fig.show()

## 7 — lcz_get_parameters

In [9]:
# Full 35-band parameter stack
params = lcz_get_parameters(map_path, istack=True, isave=True)
print('stack shape:', params.array.shape)  # (35, H, W)

stack shape: (35, 221, 155)


In [10]:
# Single parameter selection
svf = lcz_get_parameters(map_path, iselect=["svf_mean"])
print('SVF mean shape:', svf.shape)  # (1, H, W)

SVF mean shape: (1, 221, 155)


## 8 — lcz_plot_parameters
Pass a 2-D band slice directly (no intermediate file needed).

In [11]:
# SVF mean (band index 24 in the 35-band stack)
BAND = {"SVFmean": 24, "AHmean": 34, "BSFmean": 26}

for name, idx in BAND.items():
    fig = lcz_plot_parameters(params.array[idx], iselect=[name], lang = "zh")
    fig.show()

## 9 — lcz_get_ucp sem estações

Quando `stations=None`, `lcz_get_ucp` não faz extração por ponto. Em vez disso, ele retorna o stack de parâmetros como `combined_rasters` e grava o GeoTIFF em `LCZ4r_output/lcz_ucp_stack.tif`. A seção abaixo usa um subconjunto estável de variáveis para manter o tutorial rápido e reprodutível, e o resultado pode ser passado diretamente para `lcz_plot_parameters`.


In [11]:
# Process the LCZ map without stations: return raster parameters for the study area mask
ucp_stack = lcz_get_ucp(
    lcz_map=map_path,
    stations=None,
    variables=["elevation", "tree", "urban"],
    process_ghsl=False,
    process_wumpod=True,
    process_vegetation=True,
    process_directional=False,
    n_workers=1,
)

print('stack_path:', ucp_stack['stack_path'])
print('df_vars is None:', ucp_stack['df_vars'] is None)
print('summary:', ucp_stack['summary'].to_dict())
print('combined_rasters vars:', list(ucp_stack['combined_rasters'].data_vars))


11:18:36 - lcz_get_ucp - INFO - ============================================================
11:18:36 - lcz_get_ucp - INFO - Urban Parameter Processor Initialized
11:18:36 - lcz_get_ucp - INFO - ============================================================
11:18:36 - lcz_get_ucp - INFO - Study Area ID: -37.174_-11.162_-37.033_-10.862
11:18:36 - lcz_get_ucp - INFO - Target CRS: EPSG:4326
11:18:36 - lcz_get_ucp - INFO - Target Shape: (334, 157)
11:18:36 - lcz_get_ucp - INFO - Workers: 1
11:18:36 - lcz_get_ucp - INFO - Cache: lcz4r_cache
11:18:36 - lcz_get_ucp - INFO - ============================================================
11:18:36 - lcz_get_ucp - INFO - 
11:18:36 - lcz_get_ucp - INFO - Processing 33 variables
11:18:36 - lcz_get_ucp - INFO - ============================================================
11:18:36 - lcz_get_ucp - INFO -   ✓ Using cached: elevation_-37.174_-11.162_-37.033_-10.862.tif
Variables:   0%|          | 0/33 [00:00<?, ?var/s, ✓ elevation]11:18:36 - lcz_get_ucp - I

stack_path: LCZ4r_output/lcz_ucp_stack.tif
df_vars is None: True
summary: {'total_variables': 33, 'successful': 3, 'failed': 30, 'total_time': 37.69996905326843, 'success_rate': 9.090909090909092, 'failed_variables': [('frc_esa', 'Read or write failed. lcz4r_cache/rasters/frc_esa_-37.174_-11.162_-37.033_-10.862.tif, band 1: IReadBlock failed at X offset 0, Y offset 0: TIFFReadEncodedTile() failed.'), ('hgt', 'Read or write failed. lcz4r_cache/rasters/hgt_-37.174_-11.162_-37.033_-10.862.tif, band 1: IReadBlock failed at X offset 0, Y offset 0: TIFFReadEncodedTile() failed.'), ('lb', 'Read or write failed. lcz4r_cache/rasters/lb_-37.174_-11.162_-37.033_-10.862.tif, band 1: IReadBlock failed at X offset 0, Y offset 0: TIFFReadEncodedTile() failed.'), ('lc', 'Read or write failed. lcz4r_cache/rasters/lc_-37.174_-11.162_-37.033_-10.862.tif, band 1: IReadBlock failed at X offset 0, Y offset 0: TIFFReadEncodedTile() failed.'), ('lf', 'Read or write failed. lcz4r_cache/rasters/lf_-37.174_-11.1

In [12]:
# Plot rasters returned by lcz_get_ucp directly
# lcz_plot_parameters accepts the dict returned by lcz_get_ucp and uses its 'combined_rasters' Dataset.
for name in ['elevation', 'tree', 'urban']:
    fig = lcz_plot_parameters(ucp_stack, iselect=[name], lang='zh')
    fig.show()
